[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/39.%20The%20Channel%20Dredging%20%26%20Deepening%20Investment%20Problem/P39-Tier-8_executed.ipynb)

---


# 39. The Channel Dredging and Deepening Investment Problem

## Tier 8: Value-Aligned and Ethical Framework

### Goal and scope

This notebook demonstrates a value-aligned decision framework for channel-dredging investments. For each channel segment and review cycle, it generates candidate investment decisions, scores them against five ethical principles, evaluates stakeholder impacts, classifies compliance, and selects the highest-scoring candidate that is at least conditionally compliant.

The method is a finite candidate search: it evaluates a systematic grid of candidate values and adds randomly sampled candidates. It is not a proof of a globally optimal solution over a continuous decision space. The random candidates are not seeded, so individual runs may select different candidates. The ethical scores are illustrative quantitative proxies, not empirical measurements or a substitute for stakeholder consultation, legal review, or environmental assessment.

The nominal planning horizon is 10 years, with decisions every 90 days. The simulation runs the 40 complete 90-day intervals that fit in the 3,650-day horizon; the final review date is therefore before the horizon endpoint. The model does not represent a detailed dredging schedule, physical channel dynamics, discounting, project cash flows, or measured ecological and social outcomes.

### Inputs and outputs

**Inputs** are the principle weights and thresholds, stakeholder priorities and attributes, segment characteristics, and bounded candidate ranges for dredging depth, investment cost, and local benefit. **Outputs** include selected decisions, compliance and performance summaries, stakeholder-impact summaries, value-conflict indicators, and visualizations. Monetary values are expressed in million USD; dredging depth is expressed in metres.

### Method and mathematical model

For a candidate decision \(x\), the framework first computes a score \(s_p(x)\in[0,1]\) for each ethical principle \(p\). It then forms the weighted ethical score

$$
S(x)=\sum_{p\in P} w_p s_p(x), \qquad \sum_{p\in P}w_p=1.
$$

Here, \(P\) is the set of five principles, \(w_p\) is the principle weight, and \(s_p(x)\) is the candidate's score for that principle. In the code, these quantities are represented by `ethical_principles`, `principle.weight`, and the dictionary returned by `calculate_ethical_score`.

For stakeholder \(h\), the signed impact proxy is the priority-weighted deviation of principle scores from a neutral reference score of \(0.5\), adjusted for vulnerability:

$$
I_h(x)=\left(1+0.5v_h\right)
\frac{\sum_{p\in P}q_{hp}\left(s_p(x)-0.5\right)}{\sum_{p\in P}q_{hp}}.
$$

Here, \(q_{hp}\) is stakeholder \(h\)'s priority for principle \(p\), and \(v_h\in[0,1]\) is that stakeholder's vulnerability. Positive values indicate principle scores above the neutral reference on balance; negative values indicate scores below it. This is a model-based proxy, not a measured real-world impact.

A candidate is **non-compliant** if its weighted score is below the minimum overall threshold. Otherwise, it is **conditional** if it fails any specified principle threshold or the minimum stakeholder-impact threshold; if it passes all these tests, it is **compliant**. The selected candidate solves the finite search problem

$$
\max_{x\in C} S(x)
\quad\text{subject to}\quad
\operatorname{status}(x)\in\{\text{compliant},\text{conditional}\},
$$

where \(C\) is the generated candidate set. If no candidate is accepted, the optimizer returns `None`. Ties are resolved by retaining the first candidate encountered with the best score. The search has \(5^3+20=145\) candidates per segment: 125 grid candidates and 20 random candidates. The simulation evaluates this finite set independently for each segment and review cycle; it does not use a recursive dynamic-programming formulation.

### Reading the results

A higher ethical score means a higher weighted score under the specified proxies and weights; it does not establish that a decision is ethically correct. A conditional decision passes the overall-score threshold but misses at least one additional threshold. Stakeholder-impact and equity summaries describe the model's scores, not verified outcomes. Candidate correlations can indicate how scores co-vary in the generated sample, but do not establish causal trade-offs.

### When this framework is useful

The approach is useful as a transparent teaching example when a project has multiple stakeholder groups, competing values, and explicit ethical screening criteria. In practice, the weights, proxies, thresholds, candidate ranges, and representation of affected groups require documented stakeholder participation and domain-specific validation.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
@dataclass
class EthicalPrinciple:
    """An ethical principle, its score weight, and its assessment criteria."""

    principle_id: str
    name: str
    description: str
    weight: float
    metrics: List[str]
    constraints: List[str]
    stakeholder_support: float


@dataclass
class Stakeholder:
    """A stakeholder group and its modeled priorities and attributes."""

    stakeholder_id: str
    name: str
    category: str
    values: List[str]
    priorities: Dict[str, float]
    influence: float
    vulnerability: float
    representation: float


@dataclass
class EthicalImpact:
    """A record structure for an assessed ethical impact."""

    impact_id: str
    decision_id: str
    principle_id: str
    stakeholder_id: str
    impact_type: str
    magnitude: float
    duration: str
    certainty: float
    mitigation_measures: List[str]


@dataclass
class EthicalDecision:
    """A candidate decision together with its ethical assessment."""

    decision_id: str
    timestamp: datetime
    segment_id: int
    investment_type: str
    technical_parameters: Dict[str, float]
    ethical_scores: Dict[str, float]
    stakeholder_impacts: Dict[str, float]
    overall_ethical_score: float
    compliance_status: str
    transparency_level: float


framework_start = datetime(2024, 1, 1, 8, 0, 0)
planning_horizon = timedelta(days=3650)
decision_interval = timedelta(days=90)

ethical_principles = [
    EthicalPrinciple(
        principle_id="EP001",
        name="Environmental Stewardship",
        description="Protect and preserve marine ecosystems and biodiversity.",
        weight=0.25,
        metrics=["habitat_preservation", "biodiversity_index", "water_quality", "carbon_footprint"],
        constraints=["no_net_loss_biodiversity", "emission_limits", "protected_species"],
        stakeholder_support=0.85,
    ),
    EthicalPrinciple(
        principle_id="EP002",
        name="Economic Justice",
        description="Ensure a fair distribution of costs and benefits.",
        weight=0.20,
        metrics=["cost_benefit_equity", "local_employment", "economic_accessibility", "fair_competition"],
        constraints=["minimum_wage_standards", "local_hiring_requirements", "price_reasonableness"],
        stakeholder_support=0.75,
    ),
    EthicalPrinciple(
        principle_id="EP003",
        name="Social Equity",
        description="Promote fairness and equal opportunity for affected communities.",
        weight=0.20,
        metrics=["community_impact_equity", "accessibility", "cultural_preservation", "health_safety"],
        constraints=["no_discrimination", "cultural_respect", "safety_standards", "community_consultation"],
        stakeholder_support=0.80,
    ),
    EthicalPrinciple(
        principle_id="EP004",
        name="Transparency and Accountability",
        description="Ensure open decision processes and clear responsibility assignment.",
        weight=0.15,
        metrics=["decision_transparency", "information_access", "audit_trail", "responsibility_clarity"],
        constraints=["public_disclosure", "stakeholder_participation", "independent_audit"],
        stakeholder_support=0.90,
    ),
    EthicalPrinciple(
        principle_id="EP005",
        name="Intergenerational Responsibility",
        description="Consider long-term impacts on future generations.",
        weight=0.20,
        metrics=["sustainability_index", "resource_conservation", "climate_resilience", "future_impact"],
        constraints=["sustainability_thresholds", "resource_limits", "climate_compatibility"],
        stakeholder_support=0.70,
    ),
]

stakeholders = [
    Stakeholder(
        stakeholder_id="SH001",
        name="Port Authority",
        category="economic",
        values=["efficiency", "profitability", "growth", "competitiveness"],
        priorities={"EP001": 0.3, "EP002": 0.8, "EP003": 0.6, "EP004": 0.7, "EP005": 0.5},
        influence=0.8,
        vulnerability=0.2,
        representation=0.9,
    ),
    Stakeholder(
        stakeholder_id="SH002",
        name="Environmental NGOs",
        category="environmental",
        values=["conservation", "biodiversity", "sustainability", "protection"],
        priorities={"EP001": 1.0, "EP002": 0.2, "EP003": 0.4, "EP004": 0.8, "EP005": 0.9},
        influence=0.4,
        vulnerability=0.3,
        representation=0.7,
    ),
    Stakeholder(
        stakeholder_id="SH003",
        name="Local Communities",
        category="social",
        values=["livelihood", "health", "culture", "community"],
        priorities={"EP001": 0.6, "EP002": 0.7, "EP003": 1.0, "EP004": 0.8, "EP005": 0.6},
        influence=0.5,
        vulnerability=0.8,
        representation=0.6,
    ),
    Stakeholder(
        stakeholder_id="SH004",
        name="Shipping Companies",
        category="economic",
        values=["reliability", "cost_efficiency", "safety", "market_access"],
        priorities={"EP001": 0.2, "EP002": 0.9, "EP003": 0.5, "EP004": 0.6, "EP005": 0.4},
        influence=0.7,
        vulnerability=0.4,
        representation=0.5,
    ),
    Stakeholder(
        stakeholder_id="SH005",
        name="Indigenous Groups",
        category="social",
        values=["tradition", "sovereignty", "cultural_heritage", "natural_resources"],
        priorities={"EP001": 0.9, "EP002": 0.3, "EP003": 1.0, "EP004": 0.7, "EP005": 0.8},
        influence=0.3,
        vulnerability=0.9,
        representation=0.4,
    ),
    Stakeholder(
        stakeholder_id="SH006",
        name="Future Generations",
        category="social",
        values=["sustainability", "opportunity", "environmental_health", "resource_availability"],
        priorities={"EP001": 0.8, "EP002": 0.4, "EP003": 0.7, "EP004": 0.9, "EP005": 1.0},
        influence=0.2,
        vulnerability=1.0,
        representation=0.3,
    ),
]

segments_config = {
    0: {"name": "Entrance Channel", "environmental_sensitivity": "high", "community_impact": "medium"},
    1: {"name": "Inner Channel", "environmental_sensitivity": "medium", "community_impact": "high"},
    2: {"name": "Berth Approach", "environmental_sensitivity": "low", "community_impact": "low"},
}

assert np.isclose(sum(principle.weight for principle in ethical_principles), 1.0)
assert all(0.0 <= stakeholder.vulnerability <= 1.0 for stakeholder in stakeholders)
assert all(
    set(stakeholder.priorities) == {principle.principle_id for principle in ethical_principles}
    for stakeholder in stakeholders
)

print("Value-Aligned and Ethical Framework Configuration:")
print(f"- Nominal planning horizon: {planning_horizon.days / 365:.1f} years")
print(f"- Decision interval: {decision_interval.days} days")
print(f"- Ethical principles: {len(ethical_principles)}")
print(f"- Stakeholder groups: {len(stakeholders)}")
print(f"- Channel segments: {len(segments_config)}")
print(f"- Complete decision cycles: {int(planning_horizon / decision_interval)}")

In [ ]:
class EthicalOptimizer:
    """Score candidate decisions and select the best accepted candidate."""

    def __init__(self, principles: List[EthicalPrinciple], stakeholders: List[Stakeholder]):
        self.principles = principles
        self.stakeholders = stakeholders
        self.principle_weights = {principle.principle_id: principle.weight for principle in principles}
        self.stakeholder_influence = {
            stakeholder.stakeholder_id: stakeholder.influence for stakeholder in stakeholders
        }
        self.ethical_thresholds = {
            "minimum_ethical_score": 0.7,
            "stakeholder_impact_limit": 0.3,
            "environmental_protection_minimum": 0.8,
            "social_equity_minimum": 0.6,
            "transparency_minimum": 0.8,
        }

    def calculate_ethical_score(
        self, decision_params: Dict[str, float], segment_id: int
    ) -> Dict[str, float]:
        """Map candidate parameters to scores for the five ethical principles."""
        del segment_id  # Segment characteristics adjust the feasible candidate ranges.

        dredging_depth = decision_params.get("dredging_depth", 0.0)
        environmental_impact = min(1.0, dredging_depth / 3.0)

        investment_cost = decision_params.get("investment_cost", 0.0)
        local_benefit = decision_params.get("local_benefit", 0.0)
        economic_equity = min(1.0, local_benefit / max(investment_cost, 1.0))

        community_impact = decision_params.get("community_impact", 0.0)
        accessibility = decision_params.get("accessibility", 0.5)
        social_score = (community_impact + accessibility) / 2.0

        transparency = decision_params.get("transparency", 0.7)
        accountability = decision_params.get("accountability", 0.7)
        transparency_score = (transparency + accountability) / 2.0

        sustainability = decision_params.get("sustainability", 0.6)
        long_term_impact = decision_params.get("long_term_impact", 0.5)
        intergenerational_score = (sustainability + long_term_impact) / 2.0

        return {
            "EP001": 1.0 - environmental_impact,
            "EP002": economic_equity,
            "EP003": social_score,
            "EP004": transparency_score,
            "EP005": intergenerational_score,
        }

    def calculate_stakeholder_impacts(
        self, ethical_scores: Dict[str, float], segment_id: int
    ) -> Dict[str, float]:
        """Calculate signed, vulnerability-adjusted stakeholder score proxies."""
        del segment_id

        impacts = {}
        for stakeholder in self.stakeholders:
            total_priority = sum(stakeholder.priorities.values())
            weighted_deviation = sum(
                (ethical_scores[principle_id] - 0.5) * priority
                for principle_id, priority in stakeholder.priorities.items()
            ) / total_priority
            vulnerability_factor = 1.0 + stakeholder.vulnerability * 0.5
            impacts[stakeholder.stakeholder_id] = weighted_deviation * vulnerability_factor

        return impacts

    def evaluate_ethical_compliance(
        self,
        ethical_scores: Dict[str, float],
        stakeholder_impacts: Dict[str, float],
    ) -> str:
        """Classify a candidate as compliant, conditional, or non-compliant."""
        overall_score = sum(
            ethical_scores[principle.principle_id] * principle.weight
            for principle in self.principles
        )

        if overall_score < self.ethical_thresholds["minimum_ethical_score"]:
            return "non_compliant"

        if ethical_scores["EP001"] < self.ethical_thresholds["environmental_protection_minimum"]:
            return "conditional"
        if ethical_scores["EP003"] < self.ethical_thresholds["social_equity_minimum"]:
            return "conditional"
        if ethical_scores["EP004"] < self.ethical_thresholds["transparency_minimum"]:
            return "conditional"
        if min(stakeholder_impacts.values()) < -self.ethical_thresholds["stakeholder_impact_limit"]:
            return "conditional"

        return "compliant"

    def optimize_ethical_decision(
        self,
        segment_id: int,
        constraints: Dict[str, Tuple[float, float]],
        timestamp: datetime,
    ) -> Optional[EthicalDecision]:
        """Return the highest-scoring compliant or conditional candidate."""
        best_decision = None
        best_score = -1.0

        candidates = self._generate_candidates(constraints)
        for candidate_index, candidate in enumerate(candidates):
            ethical_scores = self.calculate_ethical_score(candidate, segment_id)
            stakeholder_impacts = self.calculate_stakeholder_impacts(ethical_scores, segment_id)
            overall_score = sum(
                ethical_scores[principle.principle_id] * principle.weight
                for principle in self.principles
            )
            compliance = self.evaluate_ethical_compliance(ethical_scores, stakeholder_impacts)

            if compliance in {"compliant", "conditional"} and overall_score > best_score:
                best_score = overall_score
                best_decision = EthicalDecision(
                    decision_id=f"ED_{timestamp:%Y%m%d_%H%M}_{segment_id}_{candidate_index}",
                    timestamp=timestamp,
                    segment_id=segment_id,
                    investment_type="dredging",
                    technical_parameters=candidate,
                    ethical_scores=ethical_scores,
                    stakeholder_impacts=stakeholder_impacts,
                    overall_ethical_score=overall_score,
                    compliance_status=compliance,
                    transparency_level=candidate.get("transparency", 0.7),
                )

        return best_decision

    def _generate_candidates(
        self, constraints: Dict[str, Tuple[float, float]]
    ) -> List[Dict[str, float]]:
        """Build the systematic grid and random candidates used in the finite search."""
        candidates = []

        for dredging_depth in np.linspace(*constraints["dredging_depth"], 5):
            for investment_cost in np.linspace(*constraints["investment_cost"], 5):
                for local_benefit in np.linspace(*constraints["local_benefit"], 5):
                    candidates.append(
                        {
                            "dredging_depth": float(dredging_depth),
                            "investment_cost": float(investment_cost),
                            "local_benefit": float(local_benefit),
                            "community_impact": min(1.0, local_benefit / investment_cost * 2.0),
                            "accessibility": 0.5 + local_benefit / 10.0,
                            "transparency": 0.7,
                            "accountability": 0.7,
                            "sustainability": max(0.3, 1.0 - dredging_depth / 5.0),
                            "long_term_impact": max(0.3, 1.0 - investment_cost / 50.0),
                        }
                    )

        for _ in range(20):
            candidates.append(
                {
                    "dredging_depth": float(np.random.uniform(*constraints["dredging_depth"])),
                    "investment_cost": float(np.random.uniform(*constraints["investment_cost"])),
                    "local_benefit": float(np.random.uniform(*constraints["local_benefit"])),
                    "community_impact": float(np.random.uniform(0.3, 1.0)),
                    "accessibility": float(np.random.uniform(0.4, 1.0)),
                    "transparency": float(np.random.uniform(0.6, 1.0)),
                    "accountability": float(np.random.uniform(0.6, 1.0)),
                    "sustainability": float(np.random.uniform(0.3, 1.0)),
                    "long_term_impact": float(np.random.uniform(0.3, 1.0)),
                }
            )

        return candidates


class ValueAlignmentSystem:
    """Identify priority conflicts and summarize model-based stakeholder alignment."""

    def __init__(self, stakeholders: List[Stakeholder], principles: List[EthicalPrinciple]):
        self.stakeholders = stakeholders
        self.principles = principles
        self.value_conflicts = []
        self.alignment_history = []

    def identify_value_conflicts(self) -> List[Dict[str, Any]]:
        """Record stakeholder pairs whose priorities differ substantially."""
        conflicts = []

        for index, stakeholder1 in enumerate(self.stakeholders):
            for stakeholder2 in self.stakeholders[index + 1:]:
                max_difference = 0.0
                conflicting_principle = None

                for principle_id in stakeholder1.priorities:
                    if principle_id in stakeholder2.priorities:
                        difference = abs(
                            stakeholder1.priorities[principle_id]
                            - stakeholder2.priorities[principle_id]
                        )
                        if difference > max_difference:
                            max_difference = difference
                            conflicting_principle = principle_id

                if max_difference > 0.5:
                    principle_name = next(
                        principle.name
                        for principle in self.principles
                        if principle.principle_id == conflicting_principle
                    )
                    conflicts.append(
                        {
                            "stakeholder1": stakeholder1.name,
                            "stakeholder2": stakeholder2.name,
                            "principle": principle_name,
                            "conflict_intensity": max_difference,
                            "resolution_approach": self._suggest_resolution(
                                stakeholder1, stakeholder2
                            ),
                        }
                    )

        self.value_conflicts = conflicts
        return conflicts

    def _suggest_resolution(self, stakeholder1: Stakeholder, stakeholder2: Stakeholder) -> str:
        """Label a discussion approach using vulnerability and influence attributes."""
        total_vulnerability = stakeholder1.vulnerability + stakeholder2.vulnerability

        if total_vulnerability > 1.2:
            return "prioritize_vulnerable"
        if abs(stakeholder1.influence - stakeholder2.influence) > 0.3:
            return "influence_weighted"
        return "compromise_solution"

    def calculate_value_alignment_score(self, decision: EthicalDecision) -> float:
        """Map the weighted mean signed impact proxy onto a zero-to-one scale."""
        weighted_impacts = []
        total_weight = 0.0

        for stakeholder in self.stakeholders:
            if stakeholder.stakeholder_id in decision.stakeholder_impacts:
                weight = stakeholder.influence * 0.6 + stakeholder.representation * 0.4
                weighted_impacts.append(
                    decision.stakeholder_impacts[stakeholder.stakeholder_id] * weight
                )
                total_weight += weight

        if total_weight == 0.0:
            return 0.5

        weighted_mean = sum(weighted_impacts) / total_weight
        return float(np.clip(0.5 + weighted_mean / 1.5, 0.0, 1.0))

    def generate_stakeholder_report(
        self, decisions: List[EthicalDecision]
    ) -> Dict[str, Any]:
        """Summarize cumulative impact proxies and their cross-group dispersion."""
        report = {"stakeholder_impacts": {}, "equity_analysis": {}, "recommendations": []}

        for stakeholder in self.stakeholders:
            impacts = [
                decision.stakeholder_impacts[stakeholder.stakeholder_id]
                for decision in decisions
                if stakeholder.stakeholder_id in decision.stakeholder_impacts
            ]
            cumulative_impact = float(sum(impacts))
            report["stakeholder_impacts"][stakeholder.name] = {
                "cumulative_impact": cumulative_impact,
                "positive_decisions": sum(impact > 0 for impact in impacts),
                "negative_decisions": sum(impact < 0 for impact in impacts),
                "net_balance": cumulative_impact,
                "vulnerability_weighted_impact": cumulative_impact * (1.0 + stakeholder.vulnerability),
            }

        all_impacts = [
            data["net_balance"] for data in report["stakeholder_impacts"].values()
        ]
        if all_impacts:
            impact_range = max(all_impacts) - min(all_impacts)
            impact_std = float(np.std(all_impacts))
            equity_score = 1.0 if np.isclose(impact_range, 0.0) else 1.0 - impact_std / impact_range
            report["equity_analysis"] = {
                "impact_range": impact_range,
                "impact_std": impact_std,
                "equity_score": float(np.clip(equity_score, 0.0, 1.0)),
                "most_benefited": max(
                    report["stakeholder_impacts"],
                    key=lambda name: report["stakeholder_impacts"][name]["net_balance"],
                ),
                "most_affected": min(
                    report["stakeholder_impacts"],
                    key=lambda name: report["stakeholder_impacts"][name]["net_balance"],
                ),
            }

        return report


print("Ethical optimizer and value-alignment classes are defined.")

In [ ]:
class EthicalFramework:
    """Coordinate candidate optimization and reporting across segments and review cycles."""

    def __init__(
        self,
        principles: List[EthicalPrinciple],
        stakeholders: List[Stakeholder],
        segments_config: Dict[int, Dict[str, str]],
        framework_start: datetime,
        planning_horizon: timedelta,
        decision_interval: timedelta,
    ):
        self.principles = principles
        self.stakeholders = stakeholders
        self.segments_config = segments_config
        self.ethical_optimizer = EthicalOptimizer(principles, stakeholders)
        self.value_alignment = ValueAlignmentSystem(stakeholders, principles)
        self.framework_start = framework_start
        self.planning_horizon = planning_horizon
        self.decision_interval = decision_interval
        self.current_time = framework_start

        self.ethical_decisions = []
        self.compliance_history = []
        self.stakeholder_reports = []
        self.value_conflicts = []
        self.ethical_performance = []
        self.equity_metrics = []
        self.transparency_metrics = []
        self.base_constraints = {
            "dredging_depth": (0.5, 3.0),
            "investment_cost": (5.0, 50.0),
            "local_benefit": (2.0, 20.0),
        }

    def simulate_ethical_decision_cycle(self) -> Dict[str, Any]:
        """Optimize one candidate decision for each segment at the current review date."""
        cycle_results = {
            "date": self.current_time.date(),
            "decisions": [],
            "compliance_summary": {},
            "value_alignment_score": 0.0,
            "ethical_performance": {},
        }

        for segment_id in self.segments_config:
            constraints = self._adjust_constraints_for_segment(segment_id)
            decision = self.ethical_optimizer.optimize_ethical_decision(
                segment_id, constraints, self.current_time
            )
            if decision is not None:
                self.ethical_decisions.append(decision)
                cycle_results["decisions"].append(decision)

        decisions = cycle_results["decisions"]
        if decisions:
            cycle_results["value_alignment_score"] = float(
                np.mean(
                    [self.value_alignment.calculate_value_alignment_score(decision) for decision in decisions]
                )
            )

            total_decisions = len(decisions)
            cycle_results["compliance_summary"] = {
                "compliant_rate": sum(d.compliance_status == "compliant" for d in decisions)
                / total_decisions,
                "conditional_rate": sum(d.compliance_status == "conditional" for d in decisions)
                / total_decisions,
                "non_compliant_rate": sum(d.compliance_status == "non_compliant" for d in decisions)
                / total_decisions,
                "average_ethical_score": float(
                    np.mean([decision.overall_ethical_score for decision in decisions])
                ),
            }

        cycle_results["ethical_performance"] = self._calculate_cycle_performance(decisions)
        self.compliance_history.append(cycle_results["compliance_summary"])
        self.ethical_performance.append(cycle_results["ethical_performance"])
        self.current_time += self.decision_interval
        return cycle_results

    def _adjust_constraints_for_segment(
        self, segment_id: int
    ) -> Dict[str, Tuple[float, float]]:
        """Apply the segment's sensitivity categories to the candidate ranges."""
        constraints = self.base_constraints.copy()
        segment_info = self.segments_config[segment_id]

        if segment_info["environmental_sensitivity"] == "high":
            lower, upper = constraints["dredging_depth"]
            constraints["dredging_depth"] = (lower, upper * 0.7)
            lower, upper = constraints["investment_cost"]
            constraints["investment_cost"] = (lower * 1.2, upper)
        elif segment_info["environmental_sensitivity"] == "low":
            lower, upper = constraints["dredging_depth"]
            constraints["dredging_depth"] = (lower, upper * 1.2)

        if segment_info["community_impact"] == "high":
            lower, upper = constraints["local_benefit"]
            constraints["local_benefit"] = (lower * 1.3, upper * 1.3)

        return constraints

    def _calculate_cycle_performance(
        self, decisions: List[EthicalDecision]
    ) -> Dict[str, Any]:
        """Summarize ethical scores, impact dispersion, and transparency for a cycle."""
        if not decisions:
            return {
                "overall_score": 0.0,
                "equity_score": 0.0,
                "transparency_score": 0.0,
                "principle_scores": {},
            }

        overall_score = float(
            np.mean([decision.overall_ethical_score for decision in decisions])
        )
        all_impacts = [
            impact
            for decision in decisions
            for impact in decision.stakeholder_impacts.values()
        ]
        impact_range = max(all_impacts) - min(all_impacts)
        equity_score = (
            1.0
            if np.isclose(impact_range, 0.0)
            else 1.0 - float(np.std(all_impacts)) / impact_range
        )
        transparency_score = float(
            np.mean([decision.transparency_level for decision in decisions])
        )
        principle_scores = {
            principle.name: float(
                np.mean(
                    [decision.ethical_scores[principle.principle_id] for decision in decisions]
                )
            )
            for principle in self.principles
        }

        return {
            "overall_score": overall_score,
            "equity_score": float(np.clip(equity_score, 0.0, 1.0)),
            "transparency_score": transparency_score,
            "principle_scores": principle_scores,
        }

    def run_ethical_framework_simulation(self) -> Dict[str, Any]:
        """Run all complete review cycles within the specified planning horizon."""
        total_cycles = int(self.planning_horizon / self.decision_interval)
        print("=" * 70)
        print("VALUE-ALIGNED AND ETHICAL FRAMEWORK SIMULATION")
        print("=" * 70)
        print(f"\nNominal planning horizon: {self.planning_horizon.days / 365:.1f} years")
        print(f"Ethical principles: {len(self.principles)}")
        print(f"Stakeholder groups: {len(self.stakeholders)}")
        print(f"Channel segments: {len(self.segments_config)}")
        print(f"Complete decision cycles: {total_cycles}")

        self.value_conflicts = self.value_alignment.identify_value_conflicts()
        print(f"\nPriority conflicts identified: {len(self.value_conflicts)}")

        framework_results = []
        for cycle in range(total_cycles):
            cycle_result = self.simulate_ethical_decision_cycle()
            framework_results.append(cycle_result)

            if (cycle + 1) % 4 == 0 or cycle == total_cycles - 1:
                compliance = cycle_result["compliance_summary"]
                performance = cycle_result["ethical_performance"]
                print(
                    f"Cycle {cycle + 1:2d}: {cycle_result['date']} - "
                    f"Compliant: {compliance.get('compliant_rate', 0.0):.1%}, "
                    f"Ethical score: {performance.get('overall_score', 0.0):.3f}, "
                    f"Equity proxy: {performance.get('equity_score', 0.0):.3f}"
                )

        stakeholder_report = self.value_alignment.generate_stakeholder_report(
            self.ethical_decisions
        )
        analysis = self._analyze_framework_results(framework_results)
        return {
            "framework_results": framework_results,
            "analysis": analysis,
            "stakeholder_report": stakeholder_report,
            "value_conflicts": self.value_conflicts,
            "total_decisions": len(self.ethical_decisions),
            "final_compliance_rate": analysis["final_compliance_rate"],
        }

    def _analyze_framework_results(self, results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Aggregate cycle-level metrics and principle score trends."""
        dates = [result["date"] for result in results]
        compliance_rates = [
            result["compliance_summary"].get("compliant_rate", 0.0) for result in results
        ]
        ethical_scores = [
            result["ethical_performance"].get("overall_score", 0.0) for result in results
        ]
        equity_scores = [
            result["ethical_performance"].get("equity_score", 0.0) for result in results
        ]
        transparency_scores = [
            result["ethical_performance"].get("transparency_score", 0.0)
            for result in results
        ]
        value_alignment_scores = [
            result["value_alignment_score"] for result in results
        ]

        principle_performance = {}
        for principle in self.principles:
            scores = [
                result["ethical_performance"].get("principle_scores", {}).get(
                    principle.name, 0.0
                )
                for result in results
            ]
            principle_performance[principle.name] = {
                "scores": scores,
                "average": float(np.mean(scores)) if scores else 0.0,
                "trend": scores[-1] - scores[0] if len(scores) > 1 else 0.0,
            }

        def mean_or_zero(values: List[float]) -> float:
            return float(np.mean(values)) if values else 0.0

        return {
            "dates": dates,
            "compliance_rates": compliance_rates,
            "ethical_scores": ethical_scores,
            "equity_scores": equity_scores,
            "transparency_scores": transparency_scores,
            "value_alignment_scores": value_alignment_scores,
            "overall_compliance": mean_or_zero(compliance_rates),
            "overall_ethical_score": mean_or_zero(ethical_scores),
            "overall_equity": mean_or_zero(equity_scores),
            "overall_transparency": mean_or_zero(transparency_scores),
            "overall_value_alignment": mean_or_zero(value_alignment_scores),
            "principle_performance": principle_performance,
            "final_compliance_rate": compliance_rates[-1] if compliance_rates else 0.0,
        }


print("Ethical framework class is defined.")

In [ ]:
ethical_framework = EthicalFramework(
    ethical_principles,
    stakeholders,
    segments_config,
    framework_start,
    planning_horizon,
    decision_interval,
)
framework_results = ethical_framework.run_ethical_framework_simulation()

analysis = framework_results["analysis"]
stakeholder_report = framework_results["stakeholder_report"]

assert len(framework_results["framework_results"]) == int(
    planning_horizon / decision_interval
)
assert framework_results["total_decisions"] == len(ethical_framework.ethical_decisions)
assert all(
    decision.compliance_status in {"compliant", "conditional"}
    for decision in ethical_framework.ethical_decisions
)

print("\n" + "=" * 60)
print("SIMULATION SUMMARY")
print("=" * 60)
print(f"Selected decisions: {framework_results['total_decisions']}")
print(f"Mean compliant rate: {analysis['overall_compliance']:.1%}")
print(f"Mean weighted ethical score: {analysis['overall_ethical_score']:.3f}")
print(f"Mean equity proxy: {analysis['overall_equity']:.3f}")
print(f"Mean transparency score: {analysis['overall_transparency']:.3f}")
print(f"Mean value-alignment proxy: {analysis['overall_value_alignment']:.3f}")

print("\nPriority-conflict examples:")
for conflict in framework_results["value_conflicts"][:5]:
    print(
        f"- {conflict['stakeholder1']} and {conflict['stakeholder2']}: "
        f"{conflict['principle']} (priority difference: "
        f"{conflict['conflict_intensity']:.2f}; discussion label: "
        f"{conflict['resolution_approach']})"
    )

print("\nCumulative stakeholder impact proxies:")
for name, impact_data in stakeholder_report["stakeholder_impacts"].items():
    print(
        f"- {name}: {impact_data['net_balance']:+.3f} "
        f"(positive review scores: {impact_data['positive_decisions']}, "
        f"negative review scores: {impact_data['negative_decisions']})"
    )

print("\nMean score by ethical principle:")
for principle_name, performance in analysis["principle_performance"].items():
    print(f"- {principle_name}: {performance['average']:.3f}")

equity = stakeholder_report.get("equity_analysis", {})
if equity:
    print("\nCumulative impact-proxy dispersion:")
    print(f"- Equity proxy: {equity['equity_score']:.3f}")
    print(f"- Impact-proxy range: {equity['impact_range']:.3f}")
    print(f"- Highest cumulative proxy: {equity['most_benefited']}")
    print(f"- Lowest cumulative proxy: {equity['most_affected']}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Value-Aligned and Ethical Framework Analysis", fontsize=16, fontweight="bold")

years = [
    (date - framework_start.date()).days / 365.0
    for date in analysis["dates"]
]

ax1 = axes[0, 0]
ax1.plot(years, np.array(analysis["compliance_rates"]) * 100, color="green", marker="o", markersize=3)
ax1.set_xlabel("Years from planning start (years)")
ax1.set_ylabel("Compliant decisions (%)")
ax1.set_title("Compliance by Review Cycle")
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(years, np.array(analysis["ethical_scores"]) * 100, color="blue", marker="s", markersize=3)
ax2.set_xlabel("Years from planning start (years)")
ax2.set_ylabel("Weighted ethical score (%)")
ax2.set_title("Weighted Ethical Score by Review Cycle")
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
ax3.plot(years, np.array(analysis["equity_scores"]) * 100, label="Equity proxy", color="purple", marker="^")
ax3.plot(years, np.array(analysis["transparency_scores"]) * 100, label="Transparency score", color="orange", marker="v")
ax3.set_xlabel("Years from planning start (years)")
ax3.set_ylabel("Score (%)")
ax3.set_title("Equity Proxy and Transparency")
ax3.legend()
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 0]
stakeholder_names = list(stakeholder_report["stakeholder_impacts"])
net_impacts = [
    stakeholder_report["stakeholder_impacts"][name]["net_balance"]
    for name in stakeholder_names
]
impact_colors = ["green" if impact >= 0 else "red" for impact in net_impacts]
bars = ax4.barh(stakeholder_names, net_impacts, color=impact_colors, alpha=0.75, edgecolor="black")
ax4.axvline(0, color="black", linewidth=0.8)
ax4.set_xlabel("Cumulative signed impact proxy (score units)")
ax4.set_title("Cumulative Stakeholder Impact Proxies")
ax4.grid(True, axis="x", alpha=0.3)
for bar, impact in zip(bars, net_impacts):
    offset = 0.01 if impact >= 0 else -0.01
    ax4.text(
        impact + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{impact:+.2f}",
        ha="left" if impact >= 0 else "right",
        va="center",
    )

ax5 = axes[1, 1]
principle_names = list(analysis["principle_performance"])
principle_averages = [
    analysis["principle_performance"][name]["average"] for name in principle_names
]
bars = ax5.barh(principle_names, principle_averages, color="steelblue", alpha=0.8, edgecolor="black")
ax5.set_xlabel("Mean principle score (0–1)")
ax5.set_title("Mean Performance by Ethical Principle")
ax5.set_xlim(0, 1)
ax5.grid(True, axis="x", alpha=0.3)
for bar, average in zip(bars, principle_averages):
    ax5.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{average:.3f}",
        ha="left",
        va="center",
    )

ax6 = axes[1, 2]
ax6.plot(
    years,
    np.array(analysis["value_alignment_scores"]) * 100,
    color="darkgreen",
    marker="D",
    markersize=3,
)
ax6.set_xlabel("Years from planning start (years)")
ax6.set_ylabel("Value-alignment proxy (%)")
ax6.set_title("Stakeholder Value-Alignment Proxy")
ax6.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(
    "The panels report cycle-level score proxies, cumulative stakeholder impact proxies, "
    "and average principle scores. They describe this model and its generated candidates, "
    "not observed project outcomes."
)

In [ ]:
def analyze_ethical_tradeoffs(
    decisions: List[EthicalDecision],
    principles: List[EthicalPrinciple],
) -> Dict[str, Any]:
    """Describe pairwise correlations among principle scores in selected decisions."""
    if not decisions:
        return {"correlation_matrix": np.empty((0, 0)), "principle_names": [], "tradeoffs": []}

    principle_scores = {
        principle.name: [
            decision.ethical_scores[principle.principle_id] for decision in decisions
        ]
        for principle in principles
    }
    principle_names = list(principle_scores)
    correlation_matrix = np.eye(len(principle_names))

    for i, principle1 in enumerate(principle_names):
        for j in range(i + 1, len(principle_names)):
            principle2 = principle_names[j]
            correlation = np.corrcoef(
                principle_scores[principle1], principle_scores[principle2]
            )[0, 1]
            if not np.isfinite(correlation):
                correlation = 0.0
            correlation_matrix[i, j] = correlation
            correlation_matrix[j, i] = correlation

    tradeoffs = []
    for i in range(len(principle_names)):
        for j in range(i + 1, len(principle_names)):
            correlation = correlation_matrix[i, j]
            if correlation < -0.3:
                tradeoffs.append(
                    {
                        "principle1": principle_names[i],
                        "principle2": principle_names[j],
                        "correlation": float(correlation),
                        "tradeoff_strength": abs(float(correlation)),
                    }
                )
    tradeoffs.sort(key=lambda item: item["tradeoff_strength"], reverse=True)
    return {
        "correlation_matrix": correlation_matrix,
        "principle_names": principle_names,
        "tradeoffs": tradeoffs,
    }


tradeoff_analysis = analyze_ethical_tradeoffs(
    ethical_framework.ethical_decisions, ethical_principles
)

print("ETHICAL TRADE-OFF AND OPTIMIZATION ANALYSIS")
print("\nPairwise score correlations:")
if tradeoff_analysis["tradeoffs"]:
    print(f"Negative correlations below the selected reporting threshold: {len(tradeoff_analysis['tradeoffs'])}")
    for tradeoff in tradeoff_analysis["tradeoffs"][:3]:
        print(
            f"- {tradeoff['principle1']} and {tradeoff['principle2']}: "
            f"correlation {tradeoff['correlation']:.3f}"
        )
else:
    print("No pairwise correlation below the selected reporting threshold was identified.")
print("These correlations describe selected candidate scores; they do not establish causation.")

decisions = ethical_framework.ethical_decisions
if decisions:
    compliant_count = sum(decision.compliance_status == "compliant" for decision in decisions)
    conditional_count = sum(
        decision.compliance_status == "conditional" for decision in decisions
    )
    decision_scores = [decision.overall_ethical_score for decision in decisions]
    print("\nSelected-decision summary:")
    print(f"- Selected decisions: {len(decisions)}")
    print(f"- Compliant: {compliant_count} ({compliant_count / len(decisions):.1%})")
    print(f"- Conditional: {conditional_count} ({conditional_count / len(decisions):.1%})")
    print(f"- Mean weighted ethical score: {np.mean(decision_scores):.3f}")
    print(f"- Minimum and maximum scores: {min(decision_scores):.3f}, {max(decision_scores):.3f}")
    print(f"- Score standard deviation: {np.std(decision_scores):.3f}")
else:
    print("No candidate was selected; there are no selected decisions to summarize.")

print("\nStakeholder impact-proxy summary:")
for stakeholder in stakeholders:
    impacts = [
        decision.stakeholder_impacts[stakeholder.stakeholder_id]
        for decision in decisions
        if stakeholder.stakeholder_id in decision.stakeholder_impacts
    ]
    if impacts:
        print(f"- {stakeholder.name}: mean signed proxy {np.mean(impacts):+.3f}")

if analysis["ethical_scores"]:
    initial_score = analysis["ethical_scores"][0]
    final_score = analysis["ethical_scores"][-1]
    change = final_score - initial_score
    print("\nChange between first and last cycle mean score:")
    print(f"- First cycle: {initial_score:.3f}")
    print(f"- Last cycle: {final_score:.3f}")
    print(f"- Difference (last minus first): {change:+.3f}")
    print("This comparison is descriptive and is not evidence of a causal improvement.")
else:
    print("\nNo cycle scores are available for a first-to-last comparison.")

## Summary and interpretation

This notebook connects stakeholder priorities and ethical principles to a finite candidate-search procedure. The optimizer evaluates each candidate using weighted principle scores, checks overall and principle-specific thresholds, evaluates signed stakeholder impact proxies, and retains the highest-scoring candidate classified as compliant or conditional. The simulation repeats this process for each channel segment and review cycle.

The reported compliance rates, principle scores, value-alignment scores, and equity measures are conditional on the illustrative formulas, weights, thresholds, candidate ranges, and sampled candidates defined in the notebook. Because random candidates are included without a fixed seed, results can vary between runs. The selected candidate is best only within the generated finite candidate set; the method does not certify a continuous-space global optimum.

Priority differences are indicators for structured discussion, not evidence that a conflict has been resolved. Similarly, the stakeholder impact proxies and score correlations are analytical summaries, not measured effects on communities or ecosystems. Real investment decisions would require validated data, explicit treatment of uncertainty and distributional effects, stakeholder participation, and applicable environmental and legal assessments.

The framework is therefore best understood as a transparent decision-support demonstration: it shows how ethical criteria can be represented and screened within an optimization workflow while making the assumptions and limits of that representation visible.